### 1. 패키지 설치 + 환경변수 로드

In [1]:
%pip install -qU langchain langchain_openai langgraph

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 2. 그래프 준비

체크포인트 구조에 집중하기 위해 도구 없이 단일 노드 그래프로 축소함

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

llm = ChatOpenAI(model="gpt-4o-mini")

def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

memory = InMemorySaver()
graph = graph_builder.compile(checkpointer=memory)

### 3. 대화 두 번 진행

In [4]:
from langchain_core.runnables import RunnableConfig

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

graph.invoke({"messages": [("user", "내 이름은 길동이야. 기억해줘")]}, config)
graph.invoke({"messages": [("user", "내 이름이 뭐라고 했지?")]}, config)

print("대화 완료")

대화 완료


### 4. StateSnapshot 구조

`get_state()` 가 돌려주는 값의 필드를 하나씩 확인함

| 필드 | 의미 |
|---|---|
| `values` | 이 체크포인트 시점의 State 값 |
| `next` | 다음에 실행할 노드 이름. 비어있으면 `()` → 실행 완료 |
| `config` | thread_id, checkpoint_ns, **checkpoint_id** |
| `metadata` | `source`(input/loop/update), `writes`(노드 출력), `step` |
| `created_at` | ISO 8601 생성 시각 |
| `parent_config` | 직전 체크포인트의 config. 최초는 `None` |
| `tasks` | 이 스텝에서 실행할 작업 |

In [5]:
snapshot = graph.get_state(config)

print("values 메시지 수:", len(snapshot.values["messages"]))
print("next:", snapshot.next)
print("config:", snapshot.config)
print("created_at:", snapshot.created_at)
print("parent_config:", snapshot.parent_config)
print("tasks:", snapshot.tasks)

values 메시지 수: 4
next: ()
config: {'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18db4b-be23-6446-8004-18d3d1ee3f8c'}}
created_at: 2026-08-01T14:24:41.098963+00:00
parent_config: {'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f18db4b-b31b-69ef-8003-7854923b93a9'}}
tasks: ()


### 5. metadata 확인

서드파티 유틸 없이 `json.dumps` 로 구조를 펼쳐 봄

In [6]:
import json

print(json.dumps(snapshot, ensure_ascii=False, indent=2, default=str))

[
  {
    "messages": [
      "content='내 이름은 길동이야. 기억해줘' additional_kwargs={} response_metadata={} id='86523683-dd9f-4e6c-8035-23aa72712f68'",
      "content='안녕하세요, 길동님! 이름을 기억하고 있습니다. 무엇을 도와드릴까요?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 18, 'total_tokens': 39, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ab0a2ab924', 'id': 'chatcmpl-E84vgscTTdfhsLRsIUgTYI9cncgqL', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019fbdb6-7126-7f81-8c52-ff143e5642a5-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 18, 'output_tokens': 21, 'total_tokens': 39, 'input_token_details': {'audio': 0, 'ca

### 6. checkpoint_id

`config` 안의 `checkpoint_id` 가 이 시점을 가리키는 주소임. 05번 time travel에서 이 값을 사용함

In [7]:
parent_config = snapshot.parent_config or {}

print("현재 checkpoint_id:", snapshot.config.get("configurable", {}).get("checkpoint_id"))
print("직전 checkpoint_id:", parent_config.get("configurable", {}).get("checkpoint_id"))

현재 checkpoint_id: 1f18db4b-be23-6446-8004-18d3d1ee3f8c
직전 checkpoint_id: 1f18db4b-b31b-69ef-8003-7854923b93a9


### 7. get_state_history — 스텝별 체크포인트 나열

최신 것이 먼저 나옴

In [8]:
history = list(graph.get_state_history(config))

print(f"체크포인트 {len(history)}개\n")
for state in history:
    metadata = state.metadata or {}
    checkpoint_id = state.config.get("configurable", {}).get("checkpoint_id")
    print(f"step={metadata.get('step', ''):>2} "
          f"source={metadata.get('source', ''):<6} "
          f"next={str(state.next):<12} "
          f"messages={len(state.values.get('messages', []))} "
          f"id={checkpoint_id}")

체크포인트 6개

step= 4 source=loop   next=()           messages=4 id=1f18db4b-be23-6446-8004-18d3d1ee3f8c
step= 3 source=loop   next=('chatbot',) messages=3 id=1f18db4b-b31b-69ef-8003-7854923b93a9
step= 2 source=input  next=('__start__',) messages=2 id=1f18db4b-b319-62e4-8002-3572606c1149
step= 1 source=loop   next=()           messages=2 id=1f18db4b-b316-6f76-8001-3fe1e0bc4db7
step= 0 source=loop   next=('chatbot',) messages=1 id=1f18db4b-a24d-6eb2-8000-f3a2f9e4a2e4
step=-1 source=input  next=('__start__',) messages=0 id=1f18db4b-a249-6076-bfff-a793c8165456


### 8. 정리

- 체크포인트는 **스텝마다** 쌓이며, `get_state()` 는 그중 가장 최신 것을 돌려줌
- `checkpoint_id` 로 특정 시점을 지목할 수 있음 → 05번 time travel의 전제
- `metadata.writes` 에는 그 스텝에서 각 노드가 무엇을 반환했는지가 담김
- 참고: [Checkpointers](https://docs.langchain.com/oss/python/langgraph/checkpointers)